In [5]:
import numpy as np
import tensorflow as tf
from keras.datasets import cifar10
from keras.utils import np_utils
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from art.attacks.evasion import SaliencyMapMethod
from art.estimators.classification  import KerasClassifier
from sklearn.metrics import accuracy_score

In [6]:
tf.compat.v1.disable_eager_execution()

In [7]:
import tensorflow as tf
import json
# download mnist data and split into train and test sets
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
# reshape data to fit model
X_train, X_test = X_train/255, X_test/255
# one-hot encode target column
y_train = tf.keras.utils.to_categorical(y_train)
y_test = tf.keras.utils.to_categorical(y_test)
# create model
model = Sequential()
model.add(Conv2D(32, (3, 3), padding='same', input_shape=(32, 32, 3), activation='relu'))
model.add(Conv2D(32, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(10, activation='softmax'))

In [8]:
# compile model using accuracy as a measure of model performance
model.compile(optimizer='adam', loss='categorical_crossentropy',
              metrics=['accuracy'])

# train model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5)

json.dump({'model': model.to_json()}, open("model.json", "w"))
model.save_weights("model_weights.h5")


Train on 50000 samples, validate on 10000 samples
Epoch 1/5
50000/50000 [==============================] - 92s 2ms/sample - loss: 1.5139 - accuracy: 0.4482 - val_loss: 1.1562 - val_accuracy: 0.5882
Epoch 2/5
50000/50000 [==============================] - 90s 2ms/sample - loss: 1.1277 - accuracy: 0.5998 - val_loss: 0.9704 - val_accuracy: 0.6616
Epoch 3/5
50000/50000 [==============================] - 89s 2ms/sample - loss: 0.9657 - accuracy: 0.6598 - val_loss: 0.8249 - val_accuracy: 0.7163
Epoch 4/5
50000/50000 [==============================] - 89s 2ms/sample - loss: 0.8768 - accuracy: 0.6924 - val_loss: 0.7907 - val_accuracy: 0.7253
Epoch 5/5
50000/50000 [==============================] - 91s 2ms/sample - loss: 0.8048 - accuracy: 0.7153 - val_loss: 0.7460 - val_accuracy: 0.7406


In [9]:
#Create a KerasClassifier for the model
classifier = KerasClassifier(model=model, clip_values=(0, 1),  use_logits=False)
# Generate adversarial examples
x_test = X_test # your test data
y = y_test # the true labels for the test data
jsma = SaliencyMapMethod(classifier=classifier, theta=1, gamma=0.7)
#x_test_adv = jsma.generate(x=x_test, y=y)
x_test_adv = jsma.generate(x_test)

C:\Users\nidhi.jain1\Anaconda3\lib\site-packages\keras\engine\training_v1.py:2357: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


JSMA:   0%|          | 0/10000 [00:00<?, ?it/s]

In [10]:
# Evaluate the model's accuracy before adv attack on the test dataset
score1 = model.evaluate(x_test, y_test, verbose=0)
print('Test accuracy:', score1[1])

# Evaluate the model's accuracy on the test dataset
model.fit(X_train, y_train, batch_size=32, epochs=5, validation_data=(x_test_adv, y_test))
score1 = model.evaluate(x_test_adv, y_test, verbose=0)
print('Test accuracy:', score1[1])

x_test_adv
x_test
print(f"X_train shape: {x_test_adv.shape}")
print(f"y_train shape: {x_test.shape}")

Test accuracy: 0.7406
Train on 50000 samples, validate on 10000 samples
Epoch 1/5
50000/50000 [==============================] - 93s 2ms/sample - loss: 0.7595 - accuracy: 0.7327 - val_loss: 1.6871 - val_accuracy: 0.4015
Epoch 2/5
50000/50000 [==============================] - 92s 2ms/sample - loss: 0.7239 - accuracy: 0.7452 - val_loss: 1.7984 - val_accuracy: 0.3434
Epoch 3/5
50000/50000 [==============================] - 95s 2ms/sample - loss: 0.6974 - accuracy: 0.7561 - val_loss: 1.8676 - val_accuracy: 0.3638
Epoch 4/5
50000/50000 [==============================] - 92s 2ms/sample - loss: 0.6623 - accuracy: 0.7665 - val_loss: 1.8436 - val_accuracy: 0.3886
Epoch 5/5
50000/50000 [==============================] - 92s 2ms/sample - loss: 0.6369 - accuracy: 0.7757 - val_loss: 1.9263 - val_accuracy: 0.3816
Test accuracy: 0.3816
X_train shape: (10000, 32, 32, 3)
y_train shape: (10000, 32, 32, 3)
